# **Lab Assignment–2**

## **1. Implement a feedforward neural network (FNN) for a classification task with one hidden layers from scratch.**

### a) Forward propagation from scratch. (1)
### b) Function for calculating the Binary cross-entropy loss. (1)
### c) Implement Batch Gradient Descent with Backpropagation from scratch using the following neural network architecture:
    • Input layer
    • One hidden layer with 2 neurons and ReLU activation function
    • Output layer with 1 neuron and Sigmoid activation function
    
### Use Binary Cross-Entropy as the loss function and train the model on the modified Titanic Survival Prediction dataset. 

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

#Reading from the csv files
df= pd.read_csv("titanic_selected_features.csv")

#Extracting X and y from the imported dataframe and choosing the Survived feature as target
X = df.iloc[:, 1:].values
y =df.iloc[:, 0].values.reshape(-1,1)

#We are doing normalization to scale features to a similar range
scaler = StandardScaler()
X=scaler.fit_transform(X)

#Doing train_test_split on the X and y arrays
X_train,X_temp, y_train, y_temp = train_test_split(X, y,test_size=0.3, random_state=42, stratify = y)
X_val, X_test,y_val, y_test = train_test_split(X_temp,y_temp, test_size=0.5,random_state=42, stratify = y_temp) 

In [2]:
#Defining the sigmoid and relu functions and also the loss function

def sigmoid(z):
    return 1/(1+np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)


#Here we are using BCE as it is apt for the given model
def binary_cross_entropy(y_true, y_pred):
    eps = 1e-8
    return -np.mean(y_true*np.log(y_pred+eps) + (1-y_true)*np.log(1-y_pred+eps))

In [3]:
#Creating a class to use FNN from scratch

class ScratchFNN:
    def __init__(self, layer_sizes, learning_rate=0.1):
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.params = {}
        for i in range(len(layer_sizes)-1):
            self.params["W"+str(i+1)] = np.random.randn(layer_sizes[i],layer_sizes[i+1])*0.01
            self.params["b"+str(i+1)] = np.zeros((1, layer_sizes[i+1]))

    def forward(self, X):
        cache = {"A0": X}
        L = len(self.layer_sizes)-1
        for i in range(1, L):
            Z = cache["A"+str(i-1)].dot(self.params["W"+str(i)])+self.params["b"+str(i)]
            A = relu(Z)
            cache["Z"+str(i)] = Z
            cache["A"+str(i)] = A
        ZL = cache["A"+str(L-1)].dot(self.params["W"+str(L)])+self.params["b"+str(L)]
        AL = sigmoid(ZL)
        cache["Z"+str(L)] = ZL
        cache["A"+str(L)] = AL
        return AL, cache

    def backward(self,y,cache):
        grads = {}
        L = len(self.layer_sizes)-1
        m = y.shape[0]
        AL = cache["A"+str(L)]
        dZ = AL - y
        for i in reversed(range(1, L+1)):
            A_prev = cache["A"+str(i-1)]
            grads["dW"+str(i)] = A_prev.T.dot(dZ)/m
            grads["db"+str(i)] = np.sum(dZ,axis=0,keepdims=True)/m
            if i > 1:
                dA_prev = dZ.dot(self.params["W"+str(i)].T)
                dZ = dA_prev * relu_derivative(cache["Z"+str(i-1)])
        return grads

    def update(self, grads):
        L = len(self.layer_sizes)-1
        for i in range(1, L+1):
            self.params["W"+str(i)]-=self.lr*grads["dW"+str(i)]
            self.params["b"+str(i)]-=self.lr*grads["db"+str(i)]

    def train(self, X, y, epochs=100):
        for epoch in range(epochs):
            y_pred, cache = self.forward(X)
            loss = binary_cross_entropy(y, y_pred)
            grads = self.backward(y, cache)
            self.update(grads)
            if epoch % 10 == 0:
                print(f"Epoch {epoch} Loss: {loss:.4f}")

    def predict(self, X):
        y_pred, _ = self.forward(X)
        return (y_pred > 0.5).astype(int), y_pred

## **2. Using the given dataset, perform the following tasks. First, split the dataset into training, validation, and test sets.**

### a) Perform hyperparameter tuning using the training and validation sets by experimenting with the following values:
    • Number of epochs: 10, 15, 25, 100
    • Number of hidden units: 4, 8, 16
    • Learning rate: 0.2, 0.1, 0.01
### Select the best hyperparameter combination based on validation performance. (2)

### b) Using the best hyperparameters obtained from validation, evaluate the final trained model on the test set and report the classification accuracy and loss. 

In [4]:
#These are the hyperparameters which we will use for tuning based on validation sets
epochs_list = [10,15,25,100]
hidden_units = [4,8,16]
learning_rates = [0.2,0.1,0.01]

In [5]:
#Defining the variables to store the best accuracy, configuration and model
best_acc = 0
best_config = None
best_model = None

In [17]:
from sklearn.metrics import accuracy_score

#Initializing the hyperparameter tuniung to get the best hyperparameters
input_size = X_train.shape[1]
for e in epochs_list:
    for h in hidden_units:
        for lr in learning_rates:
            print(f"Testing epochs={e}, hidden={h}, lr={lr}")
            model = ScratchFNN([input_size, h, 1],learning_rate=lr)
            model.train(X_train, y_train, epochs=e)
            preds,_ = model.predict(X_val)
            acc = accuracy_score(y_val, preds)
            print()
            if acc > best_acc:
                best_acc = acc
                best_config = (e,h,lr)
                best_model = model

#Printing the best hyperparameters
print()
print("Best Hyperparameters:", best_config)

Testing epochs=10, hidden=4, lr=0.2
Epoch 0 Loss: 0.6932

Testing epochs=10, hidden=4, lr=0.1
Epoch 0 Loss: 0.6932

Testing epochs=10, hidden=4, lr=0.01
Epoch 0 Loss: 0.6931

Testing epochs=10, hidden=8, lr=0.2
Epoch 0 Loss: 0.6931

Testing epochs=10, hidden=8, lr=0.1
Epoch 0 Loss: 0.6932

Testing epochs=10, hidden=8, lr=0.01
Epoch 0 Loss: 0.6932

Testing epochs=10, hidden=16, lr=0.2
Epoch 0 Loss: 0.6931

Testing epochs=10, hidden=16, lr=0.1
Epoch 0 Loss: 0.6932

Testing epochs=10, hidden=16, lr=0.01
Epoch 0 Loss: 0.6931

Testing epochs=15, hidden=4, lr=0.2
Epoch 0 Loss: 0.6932
Epoch 10 Loss: 0.6757

Testing epochs=15, hidden=4, lr=0.1
Epoch 0 Loss: 0.6931
Epoch 10 Loss: 0.6824

Testing epochs=15, hidden=4, lr=0.01
Epoch 0 Loss: 0.6931
Epoch 10 Loss: 0.6918

Testing epochs=15, hidden=8, lr=0.2
Epoch 0 Loss: 0.6932
Epoch 10 Loss: 0.6757

Testing epochs=15, hidden=8, lr=0.1
Epoch 0 Loss: 0.6931
Epoch 10 Loss: 0.6823

Testing epochs=15, hidden=8, lr=0.01
Epoch 0 Loss: 0.6931
Epoch 10 Loss

The loss decreases with increasing epochs as the network is learning the underlying mapping between inputs and outputs.

In [7]:
#Here we are gonna get y_pred using the best model and then calculating the test accuracy and loss

test_preds, test_probs = best_model.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)
test_loss = binary_cross_entropy(y_test, test_probs)

print("Test Accuracy of Model from Scratch:", test_acc)
print("Test Loss of Model from Scratch:", test_loss)

Test Accuracy of Model from Scratch: 0.746268656716418
Test Loss of Model from Scratch: 0.5387776685143559


## **3. Re-implement the same neural network architecture using PyTorch, keeping the activation functions and the best hyperparameters obtained in the previous part unchanged. Compare the performance of the PyTorch model with your from-scratch implementation, and discuss the observed results with proper analysis.**

In [8]:
#Importing the torch modules to implement hte FNN using the torch inbuilt modules
import torch
import torch.nn as nn
import torch.optim as optim

In [9]:
#Creating the class TorchFNN for implementing the FNN using the torch modules
class TorchFNN(nn.Module):
    def __init__(self, input_size, hidden_units):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.model(x)

In [10]:
#Extracting the epochs, hidden config and learning rate of the best configuration that we got earlier
epochs, hidden, lr = best_config
torch_model = TorchFNN(input_size, hidden)

In [11]:
#Initializing the criterion(BCE Loss) and optimizer
criterion = nn.BCELoss()
optimizer = optim.SGD(torch_model.parameters(), lr=lr)

In [12]:
#Converting the training datasets to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

In [13]:
#Running the mdoel for the given number of epochs(best config)
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = torch_model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print("Torch Epoch:", epoch, "Loss:", loss.item())

Torch Epoch: 0 Loss: 0.6982534527778625
Torch Epoch: 10 Loss: 0.5430721640586853
Torch Epoch: 20 Loss: 0.4860624372959137
Torch Epoch: 30 Loss: 0.46376538276672363
Torch Epoch: 40 Loss: 0.4547044336795807
Torch Epoch: 50 Loss: 0.4503536820411682
Torch Epoch: 60 Loss: 0.447645902633667
Torch Epoch: 70 Loss: 0.4456593096256256
Torch Epoch: 80 Loss: 0.4439528286457062
Torch Epoch: 90 Loss: 0.4423631727695465


In [14]:
#Calculating the test accuracy for the pytorch model
X_test_t = torch.tensor(X_test, dtype=torch.float32)

with torch.no_grad():
    outputs = torch_model(X_test_t)
    preds = (outputs.numpy() > 0.5).astype(int)

torch_acc = accuracy_score(y_test, preds)
print("PyTorch Test Accuracy:", torch_acc)

PyTorch Test Accuracy: 0.753731343283582


### We could see that the accuracy of the from scratch implementation is really close to the inbuilt pytorch model with the same hyperparameters. The accuracies of both the models is given below:

In [15]:
print("Accuracy of from scratch model: ",test_acc)
print("Accuracy of the pytorch model: ",torch_acc)

Accuracy of from scratch model:  0.746268656716418
Accuracy of the pytorch model:  0.753731343283582


## **4. Generalize the forward propagation and backpropagation algorithms for any number of hidden layers and any number of hidden units.**

### The class ScratchFNN has been generalized for any number of hidden layers and any number of hidden units. While calling the class/model, you just have to provide the layer configuration as attribute, for example, if the neural network has 1 hidden layer and input 4 neurons, output 1 neuron and hidden layer 2 neurons, then the inputted attribute will be [4,2,1]